# 🥗 Model 2: Ingredient Identification & Nutrition Prediction (Version 1)
**South Indian Food Nutrition & Detection AI Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dharmenthiran-spi/SouthIndian_Food_Nutrition/blob/main/Incredient_Identification/NoteBooks/Ingredient%20Identification/Version1/ingredient_identification_v1_training.ipynb)

---
### 🔄 Clean Single-Source Workflow (No Google Drive Sync Issues)
1. **Click 'Open in Colab'** badge above to launch this notebook in Google Colab.
2. **Run All Cells**: Colab freshly clones the repository and trains on ultra-fast local disk (`/content/`).
3. **1-Click Download**: Automatically downloads `best.pt` and `last.pt` directly to your local computer's **Downloads** folder.
4. **Local Integration**: Place `best.pt` in `Incredient_Identification/Models/Ingredient Identification/Version1/` and launch `streamlit run app.py`!

## Step 1: Environment & Dependency Installation
Installs required libraries and checks for GPU acceleration.

In [ ]:
# Install required dependencies
!pip install -q torch torchvision pandas numpy matplotlib pyyaml scikit-learn

import torch, sys
print("=" * 60)
print("🖥️  SYSTEM & HARDWARE SPECIFICATIONS")
print("=" * 60)
print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device       : {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (Fast & lightweight for Model 2)")
print("=" * 60)

## Step 2: Fresh Repository Clone in Colab (Fast Local Disk)
Clones the latest code directly from GitHub into Colab's fast local SSD storage.

In [ ]:
import os

REPO_URL = "https://github.com/Dharmenthiran-spi/SouthIndian_Food_Nutrition.git"
REPO_NAME = "SouthIndian_Food_Nutrition"

# In Google Colab, fresh clone
if "google.colab" in str(get_ipython()):
    if not os.path.exists(REPO_NAME):
        print(f"🚀 Freshly cloning {REPO_NAME} from GitHub...")
        !git clone {REPO_URL}
    %cd {REPO_NAME}
    print(f"Active Directory: {os.getcwd()}")
else:
    print(f"Running in local workspace: {os.getcwd()}")

# Locate dataset files
DATASET_DIR = "Incredient_Identification/DataSet/Version1 Used Data"
if not os.path.exists(DATASET_DIR):
    # Fallback search
    for candidate in ["SouthIndian_Food_Nutrition/Incredient_Identification/DataSet/Version1 Used Data", "../Incredient_Identification/DataSet/Version1 Used Data"]:
        if os.path.exists(candidate):
            DATASET_DIR = candidate
            break

print(f"✅ Verified Dataset Path: {DATASET_DIR}")

## Step 3: Load the 31 South Indian Dishes Dataset
Loads the 31 dishes, constituent ingredients, and nutritional ranges (Calories, Protein, Carbs, Fat, Fiber).

In [ ]:
import pandas as pd, json

csv_path = os.path.join(DATASET_DIR, "south_indian_food_ingredients_nutrition.csv")
vocab_path = os.path.join(DATASET_DIR, "ingredients_vocabulary.csv")

df = pd.read_csv(csv_path)
vocab_df = pd.read_csv(vocab_path)

ALL_INGREDIENTS = sorted(vocab_df["ingredient"].tolist())
NUM_INGREDIENTS = len(ALL_INGREDIENTS)
ING2IDX = {ing: i for i, ing in enumerate(ALL_INGREDIENTS)}
IDX2ING = {i: ing for i, ing in enumerate(ALL_INGREDIENTS)}

FOOD_NAMES = sorted(df["food_name"].tolist())
NUM_FOODS = len(FOOD_NAMES)
FOOD2IDX = {name: i for i, name in enumerate(FOOD_NAMES)}
IDX2FOOD = {i: name for i, name in enumerate(FOOD_NAMES)}

print(f"📊 Total Food Classes (from Food_Detection): {NUM_FOODS}")
print(f"🌿 Total Canonical Ingredients in Vocabulary: {NUM_INGREDIENTS}")
display(df[["food_id", "food_name", "portion_size", "calories_kcal", "protein_g", "carbs_g", "fat_g", "fiber_g"]].head(8))

## Step 4: Dataset Preprocessing & DataLoaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

nut_cols = ["calories_kcal", "protein_g", "carbs_g", "fat_g", "fiber_g"]
nut_mean = torch.tensor(df[nut_cols].mean().values, dtype=torch.float32)
nut_std = torch.tensor(df[nut_cols].std().values, dtype=torch.float32)

class SouthIndianIngredientNutritionDataset(Dataset):
    def __init__(self, dataframe, augment=False):
        self.df = dataframe.reset_index(drop=True)
        self.augment = augment
        
    def __len__(self):
        return len(self.df) * (12 if self.augment else 1)
    
    def __getitem__(self, idx):
        real_idx = idx % len(self.df)
        row = self.df.iloc[real_idx]
        
        food_idx = FOOD2IDX[row["food_name"]]
        food_tensor = torch.tensor(food_idx, dtype=torch.long)
        
        ing_target = torch.zeros(NUM_INGREDIENTS, dtype=torch.float32)
        for ing in str(row["main_ingredients"]).split(";"):
            ing = ing.strip()
            if ing in ING2IDX:
                ing_target[ING2IDX[ing]] = 1.0
        
        raw_nut = torch.tensor([
            row["calories_kcal"],
            row["protein_g"],
            row["carbs_g"],
            row["fat_g"],
            row["fiber_g"]
        ], dtype=torch.float32)
        
        if self.augment and idx >= len(self.df):
            noise = 1.0 + torch.randn(5) * 0.03
            raw_nut = raw_nut * noise
        
        nut_target = (raw_nut - nut_mean) / nut_std
        return food_tensor, ing_target, nut_target

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_dataset = SouthIndianIngredientNutritionDataset(train_df, augment=True)
val_dataset = SouthIndianIngredientNutritionDataset(val_df, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

## Step 5: Multi-Task Neural Network Architecture
Dense embedding and feature extractor with dual multi-task heads for ingredients and nutrition.

In [ ]:
import torch.nn as nn

class SouthIndianIngredientNutritionModel(nn.Module):
    def __init__(self, num_foods=NUM_FOODS, num_ingredients=NUM_INGREDIENTS, embed_dim=64):
        super().__init__()
        self.food_embedding = nn.Embedding(num_foods, embed_dim)
        self.shared_trunk = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        self.ingredient_head = nn.Sequential(
            nn.Linear(128, 96),
            nn.ReLU(),
            nn.Linear(96, num_ingredients)
        )
        self.nutrition_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 5)
        )
        
    def forward(self, food_ids):
        embed = self.food_embedding(food_ids)
        shared = self.shared_trunk(embed)
        return self.ingredient_head(shared), self.nutrition_head(shared)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SouthIndianIngredientNutritionModel().to(device)
print("Model initialized on:", device)

## Step 6: Multi-Task Training Loop & Checkpoint Saving
Trains the network and saves `best.pt` and `last.pt`.

In [ ]:
import torch.optim as optim

OUTPUT_DIR = "Incredient_Identification/Models/Ingredient Identification/Version1"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BEST_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, "best.pt")
LAST_WEIGHTS_PATH = os.path.join(OUTPUT_DIR, "last.pt")

criterion_ing = nn.BCEWithLogitsLoss()
criterion_nut = nn.SmoothL1Loss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

EPOCHS = 40
best_val_loss = float("inf")
history = {"train_loss": [], "val_loss": [], "ing_loss": [], "nut_loss": []}

print(f"🚀 Training Model 2 for {EPOCHS} epochs...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for food_ids, ing_targets, nut_targets in train_loader:
        food_ids, ing_targets, nut_targets = food_ids.to(device), ing_targets.to(device), nut_targets.to(device)
        optimizer.zero_grad()
        p_ing, p_nut = model(food_ids)
        loss = criterion_ing(p_ing, ing_targets) + 1.2 * criterion_nut(p_nut, nut_targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(food_ids)
    train_loss /= len(train_dataset)
    
    model.eval()
    val_loss, val_ing, val_nut = 0.0, 0.0, 0.0
    with torch.no_grad():
        for food_ids, ing_targets, nut_targets in val_loader:
            food_ids, ing_targets, nut_targets = food_ids.to(device), ing_targets.to(device), nut_targets.to(device)
            p_ing, p_nut = model(food_ids)
            l_ing = criterion_ing(p_ing, ing_targets)
            l_nut = criterion_nut(p_nut, nut_targets)
            val_ing += l_ing.item() * len(food_ids)
            val_nut += l_nut.item() * len(food_ids)
            val_loss += (l_ing.item() + 1.2 * l_nut.item()) * len(food_ids)
    val_loss /= len(val_dataset)
    scheduler.step(val_loss)
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    
    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "food2idx": FOOD2IDX,
        "ing2idx": ING2IDX,
        "nut_mean": nut_mean.tolist(),
        "nut_std": nut_std.tolist()
    }
    torch.save(ckpt, LAST_WEIGHTS_PATH)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(ckpt, BEST_WEIGHTS_PATH)
        star = " 🌟 [SAVED NEW best.pt]"
    else:
        star = ""
    if epoch % 5 == 0 or epoch == 1 or star:
        print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}{star}")

print(f"\n🎉 Training Complete! Saved weights to: {BEST_WEIGHTS_PATH}")

## Step 7: Live Inference Test (Food Name -> Ingredients & Nutrition Facts)
Tests prediction on sample dishes detected by Model 1 (`Food_Detection`).

In [ ]:
import numpy as np

checkpoint = torch.load(BEST_WEIGHTS_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

sample_dishes = ["Idly", "Dosa", "Ven Pongal", "Chicken briyani", "Sambar", "Appam"]

print("=" * 65)
print("🎯 LIVE PREDICTION: FOOD NAME -> INGREDIENTS & NUTRITION")
print("=" * 65)

for dish in sample_dishes:
    f_id = torch.tensor([FOOD2IDX[dish]], device=device)
    with torch.no_grad():
        pred_logits, pred_nut_norm = model(f_id)
        pred_probs = torch.sigmoid(pred_logits)[0].cpu().numpy()
        pred_nut = ((pred_nut_norm[0].cpu() * nut_std) + nut_mean).numpy()
    
    # Top predicted ingredients (threshold >= 0.40)
    ing_idx = np.where(pred_probs >= 0.40)[0]
    predicted_ings = [IDX2ING[i].replace('_', ' ').title() for i in ing_idx]
    
    print(f"🍽️ Detected Food : {dish}")
    print(f"🌿 Ingredients   : {', '.join(predicted_ings)}")
    print(f"⚡ Nutrition     : {pred_nut[0]:.1f} kcal | Protein: {pred_nut[1]:.1f}g | Carbs: {pred_nut[2]:.1f}g | Fat: {pred_nut[3]:.1f}g | Fiber: {pred_nut[4]:.1f}g")
    print("-" * 65)

## Step 8: 📥 1-Click Download Trained Weights to Your Local Computer
Run this cell in Google Colab to **instantly download `best.pt` directly to your local computer's Downloads folder**.

Then, move it into `Incredient_Identification/Models/Ingredient Identification/Version1/` on your PC!

In [ ]:
# 📥 1-Click Download weights with standardized naming: ingredient_identification_v1_best.pt
import os, shutil

# Standardized File Names
STANDARDIZED_BEST = "ingredient_identification_v1_best.pt"
STANDARDIZED_LAST = "ingredient_identification_v1_last.pt"

if os.path.exists(BEST_WEIGHTS_PATH):
    shutil.copy2(BEST_WEIGHTS_PATH, STANDARDIZED_BEST)
    print(f"✅ Prepared standardized model: {STANDARDIZED_BEST}")
    if "google.colab" in str(get_ipython()):
        from google.colab import files
        print(f"📥 Triggering download of {STANDARDIZED_BEST} to your computer...")
        files.download(STANDARDIZED_BEST)
    else:
        print(f"Saved locally as {STANDARDIZED_BEST}")
else:
    print(f"⚠️ Weights file not found at: {BEST_WEIGHTS_PATH}")


## Step 9: How to Test with the Streamlit App (`app.py`)
On your local PC:
1. Move the downloaded `best.pt` into `Incredient_Identification/Models/Ingredient Identification/Version1/best.pt`.
2. Run the Streamlit web app:
```bash
streamlit run app.py
```
3. Upload any food image $\rightarrow$ YOLOv8 detects the food $\rightarrow$ Model 2 renders the ingredients & nutrition cards in real time!